DMP PDF
  ↓
1. pdfplumber extraction
  ↓
2. rule-based structure detection
  ↓
3. build narrative JSON
  ↓
4. save final JSON


# Part 1 — Imports

In [73]:
from pathlib import Path
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

# Part 2 — Paths

In [74]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample10.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_narrative.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


# Part 3 — Run pdfplumber extraction

In [75]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-05 19:27:55] Extracting line-level text with pdfplumber: sample10.pdf
[2026-05-05 19:27:56] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample10.json
[2026-05-05 19:27:56] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample10.txt
Extracted lines: 68
Saved pdfplumber JSON: True


# Part 4 — Run rule-based structure detection

In [76]:

structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)
print("Detected format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].head(100)

Detected format: numbered_sections
label
content           61
section            6
document_title     1
Name: count, dtype: int64


,page,line_order,text,avg_font_size,is_bold,label,document_format
0,1,1,DATA MANAGEMENT,10.98,True,document_title,numbered_sections
1,1,2,1. Policy and Practice,10.98,True,section,numbered_sections
2,1,3,We will ensure that the data obtained and this...,10.98,False,content,numbered_sections
3,1,4,The Bourns College of Engineering (BCOE) at UC...,10.98,False,content,numbered_sections
4,1,5,"California Digital Library, has established a ...",10.98,False,content,numbered_sections
...,...,...,...,...,...,...,...
63,2,17,6. Archiving and Access,10.98,True,section,numbered_sections
64,2,18,It is understood that the NSF Engineering Dire...,10.98,False,content,numbered_sections
65,2,19,of three years after conclusion of the award o...,10.98,False,content,numbered_sections
66,2,20,"institution, the products will remain stored i...",10.98,False,content,numbered_sections


# Part 5 — Inspect extracted lines

In [77]:
print("Detected document format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

Detected document format: numbered_sections
label
content           61
section            6
document_title     1
Name: count, dtype: int64


# Part 6 — Save CSV debug file

In [78]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].to_csv(csv_output_path, index=False, encoding="utf-8")

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample10_structured_lines.csv


# Part 7 — Build narrative JSON

In [79]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

sections = final_json["narrative"]["template"]["section"]

print("Saved JSON:", final_json_path)
print("Number of sections:", len(sections))

for sec in sections:
    print(sec["order"], sec["title"], "| questions:", len(sec["question"]))

[2026-05-05 19:27:56] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample10_narrative.json
Saved JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample10_narrative.json
Number of sections: 6
1 1. Policy and Practice | questions: 0
2 2. Scope | questions: 0
3 3. Data and Metadata Format and Contents | questions: 0
4 4. Accessibility and Data Protection | questions: 0
5 5. Derivative Products | questions: 0
6 6. Archiving and Access | questions: 0


# Part 8 — Inspect one section

In [80]:
sections[0]

{'id': 'section_1',
 'title': '1. Policy and Practice',
 'description': 'We will ensure that the data obtained and this study will be made available to the research community.\nThe Bourns College of Engineering (BCOE) at UCR, in partnership with the UCR Libraries and the\nCalifornia Digital Library, has established a new, custom data management system designed to make\nresults of our research readily and reliably accessible. This system combines two services of the\nCalifornia Digital Library (CDL) for the first time: an eScholarship series, a curated site where all BCOE\ninvestigators can store and disseminate their data, and EZID permanent identifiers for long-term\nidentification andmanagement of data resources.\nAll principal investigators in BCOE are responsible for complying with sponsor policies regarding data\nmanagement and accessibility. All have sufficient computing resources to store and process the data\ngenerated in their research. Every principal investigator has the abi